<a href="https://colab.research.google.com/github/Afila1996/VKI-LS59-Turbulence-modeling/blob/main/VKI_LS59_v2_PINNs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install plaid-lib datasets meshio pyvista

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.2/166.2 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.9/201.9 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.6/145.6 MB 7.0 MB/s eta 0:00:00


In [ ]:
!pip install vedo

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 32.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for vedo: filename=vedo-2026.6.1-py3-none-any.whl size=2823124 sha256=4b987fc84b5598cf97126ff8e7cd97073a74382a79b4d37db2f3d868b31e8d2e
  Stored in directory: /root/.cache/pip/wheels/a6/87/43/56603acbb1373aa7463afd2561478d00a6d5bb3aa40403938b
Successfully built vedo


In [ ]:
!pip install git+https://github.com/PLAID-lib/plaid.git

  Cloning https://github.com/PLAID-lib/plaid.git to /tmp/pip-req-build-8us8dxxg
  Running command git clone --filter=blob:none --quiet https://github.com/PLAID-lib/plaid.git /tmp/pip-req-build-8us8dxxg
  Resolved https://github.com/PLAID-lib/plaid.git to commit a5ac97c21113cc4621a04e1fe936864524efeaf3
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.1/284.1 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 62.1 MB/s eta 0:00:00
  Created wheel for pyplaid: filename=pyplaid-0.1.13.dev23+ga5ac97c21-py3-none-any.whl size=165039 sha256=b013822967d1cabe7cd99b97122ed2d8f6007ea8fd8bcdf8cd3cbf2b9c3cf9ba
  Stored in directory: /tmp/pip-ephem-wheel-cache-t21fuuqj/wheels/2d/8c/4a/9d317c2f9d6cffb12280f3c0f0c3f65acf2403d8aead0c66f4
Successfully built pyplaid


In [ ]:
from datasets import load_dataset
import pickle
from plaid.containers.sample import Sample
import numpy as np
from scipy.interpolate import griddata
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import r2_score


In [ ]:
from plaid.bridges.huggingface_bridge import huggingface_dataset_to_plaid

dataset = load_dataset("PLAID-datasets/VKI-LS59",split="all_samples")

dataset, problem = huggingface_dataset_to_plaid(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/all_samples-00000-of-00006.parquet:   0%|          | 0.00/426M [00:00<?, ?B/s]

data/all_samples-00001-of-00006.parquet:   0%|          | 0.00/426M [00:00<?, ?B/s]

data/all_samples-00002-of-00006.parquet:   0%|          | 0.00/426M [00:00<?, ?B/s]

data/all_samples-00003-of-00006.parquet:   0%|          | 0.00/426M [00:00<?, ?B/s]

data/all_samples-00004-of-00006.parquet:   0%|          | 0.00/375M [00:00<?, ?B/s]

data/all_samples-00005-of-00006.parquet:   0%|          | 0.00/179M [00:00<?, ?B/s]

Generating all_samples split:   0%|          | 0/839 [00:00<?, ? examples/s]

Converting Hugging Face binary dataset to plaid: 100%|██████████| 839/839 [00:03<00:00, 221.37it/s]


In [ ]:
GRID_SIZE = 64
BETA      = 10.0

def make_uniform_grid(grid_size):
    xs = np.linspace(0, 1, grid_size)
    ys = np.linspace(0, 1, grid_size)
    gx, gy = np.meshgrid(xs, ys)
    return gx, gy

def interpolate_to_grid(nodes, values, grid_size=GRID_SIZE):

    xy_min = nodes.min(axis=0)
    xy_max = nodes.max(axis=0)
    nodes_norm = (nodes - xy_min) / (xy_max - xy_min + 1e-8)

    gx, gy = make_uniform_grid(grid_size)
    grid_points = np.column_stack([gx.ravel(), gy.ravel()])


    interp = griddata(nodes_norm, values, grid_points, method='cubic')
    mask   = np.isnan(interp)
    if mask.any():
        interp[mask] = griddata(nodes_norm, values, grid_points[mask], method='nearest')
    return interp.reshape(grid_size, grid_size)

def compute_chi(sdf_grid, beta=BETA):
    chi_sharp = (sdf_grid > 0).astype(np.float32)
    dist       = np.abs(sdf_grid)
    chi_smooth = np.tanh(beta * dist) * (chi_sharp - 0.5) + 0.5
    return chi_smooth.astype(np.float32)

In [ ]:
X_all = []
Y_all = []
Chi_all = []
G = GRID_SIZE

for idx in ids_train:
    sample = dataset[idx]

    nodes    = sample.get_nodes(base_name="Base_2_2")        # (N, 2)
    sdf      = sample.get_field("sdf",  base_name="Base_2_2") # (N,)
    angle_in = sample.get_scalar("angle_in")
    mach_out = sample.get_scalar("mach_out")

    ro   = sample.get_field("ro",   base_name="Base_2_2")
    rou  = sample.get_field("rou",  base_name="Base_2_2")
    rov  = sample.get_field("rov",  base_name="Base_2_2")
    roe  = sample.get_field("roe",  base_name="Base_2_2")
    mach = sample.get_field("mach", base_name="Base_2_2")


    sdf_grid  = interpolate_to_grid(nodes, sdf)
    chi_grid  = compute_chi(sdf_grid)


    gx, gy = make_uniform_grid(G)
    angle_grid    = np.full((G, G), angle_in,  dtype=np.float32)
    mach_out_grid = np.full((G, G), mach_out,  dtype=np.float32)

    X = np.stack([gx, gy, sdf_grid, angle_grid, mach_out_grid], axis=-1)


    ro_g   = interpolate_to_grid(nodes, ro)
    rou_g  = interpolate_to_grid(nodes, rou)
    rov_g  = interpolate_to_grid(nodes, rov)
    roe_g  = interpolate_to_grid(nodes, roe)
    mach_g = interpolate_to_grid(nodes, mach)
    Y = np.stack([ro_g, rou_g, rov_g, roe_g, mach_g], axis=-1)

    X_all.append(X)
    Y_all.append(Y)
    Chi_all.append(chi_grid)

X_all   = np.stack(X_all)
Y_all   = np.stack(Y_all)
Chi_all = np.stack(Chi_all)

print(f"X_all shape : {X_all.shape}")
print(f"Y_all shape : {Y_all.shape}")
print(f"Chi_all shape : {Chi_all.shape}")


X_all shape : (671, 128, 128, 5)
Y_all shape : (671, 128, 128, 5)
Chi_all shape : (671, 128, 128)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/vki_idafno_data_128'
import os
os.makedirs(SAVE_DIR, exist_ok=True)

Mounted at /content/drive


In [ ]:
np.save(f'{SAVE_DIR}/X_all.npy',   X_all)
np.save(f'{SAVE_DIR}/Y_all.npy',   Y_all)
np.save(f'{SAVE_DIR}/Chi_all.npy', Chi_all)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/vki_idafno_data'

REBUILD = False

if REBUILD:

    np.save(f'{SAVE_DIR}/X_all.npy',   X_all)
    np.save(f'{SAVE_DIR}/Y_all.npy',   Y_all)
    np.save(f'{SAVE_DIR}/Chi_all.npy', Chi_all)
else:
    X_all   = np.load(f'{SAVE_DIR}/X_all.npy')
    Y_all   = np.load(f'{SAVE_DIR}/Y_all.npy')
    Chi_all = np.load(f'{SAVE_DIR}/Chi_all.npy')


print(f"X_all shape   : {X_all.shape}")
print(f"Y_all shape   : {Y_all.shape}")
print(f"Chi_all shape : {Chi_all.shape}")

Mounted at /content/drive
X_all shape   : (671, 64, 64, 5)
Y_all shape   : (671, 64, 64, 5)
Chi_all shape : (671, 64, 64)


In [ ]:

from sklearn.model_selection import train_test_split


X_mean = X_all.mean(axis=(0, 1, 2))
X_std  = X_all.std( axis=(0, 1, 2))
Y_mean = Y_all.mean(axis=(0, 1, 2))
Y_std  = Y_all.std( axis=(0, 1, 2))



X_tr_raw, X_val_raw, Y_tr_raw, Y_val_raw, Chi_tr, Chi_val = train_test_split(
    X_all, Y_all, Chi_all, test_size=0.1, random_state=42)


X_tr  = (X_tr_raw  - X_mean) / (X_std  + 1e-8)
X_val = (X_val_raw - X_mean) / (X_std  + 1e-8)
Y_tr  = (Y_tr_raw  - Y_mean) / (Y_std  + 1e-8)
Y_val = (Y_val_raw - Y_mean) / (Y_std  + 1e-8)


print(f"  Train : {len(X_tr)}")
print(f"  Val   : {len(X_val)}")



  Train : 603
  Val   : 68


##Components

In [ ]:
class LpLoss(nn.Module):

    def __init__(self, p=2, reduction='mean'):
        super(LpLoss, self).__init__()
        self.p         = p
        self.reduction = reduction

    def forward(self, pred, target):

        B = pred.shape[0]


        pred_flat   = pred.reshape(B, -1)
        target_flat = target.reshape(B, -1)

        diff_norm   = torch.norm(pred_flat - target_flat, p=self.p, dim=1)
        target_norm = torch.norm(target_flat,             p=self.p, dim=1) + 1e-8

        loss = diff_norm / target_norm

        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        else:
            return loss






In [ ]:
class GaussianNormalizer(nn.Module):

    def __init__(self, x, eps=1e-8):
        super(GaussianNormalizer, self).__init__()

        self.register_buffer('mean', x.mean(dim=(0, 2, 3), keepdim=True))
        self.register_buffer('std',  x.std( dim=(0, 2, 3), keepdim=True))
        self.eps = eps

    def encode(self, x):

        return (x - self.mean) / (self.std + self.eps)

    def decode(self, x):

        return x * (self.std + self.eps) + self.mean



In [ ]:
class SpectralConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, modes1, modes2):
        super(SpectralConv2d, self).__init__()

        self.in_channels  = in_channels
        self.out_channels = out_channels
        self.modes1       = modes1
        self.modes2       = modes2

        # He initialisation
        fan_in = in_channels * modes1 * modes2
        std    = (2.0 / fan_in) ** 0.5
        scale  = std / (2.0 ** 0.5)

        self.weights1 = nn.Parameter(
            torch.view_as_complex(
                torch.randn(in_channels, out_channels, modes1, modes2, 2) * scale
            )
        )
        self.weights2 = nn.Parameter(
            torch.view_as_complex(
                torch.randn(in_channels, out_channels, modes1, modes2, 2) * scale
            )
        )

    def compl_mul2d(self, input, weights):
        return torch.einsum("bixy,ioxy->boxy", input, weights)

    def forward(self, x):
        batchsize = x.shape[0]

        ## FFT
        x_ft = torch.fft.rfft2(x)


        out_ft = torch.zeros(batchsize, self.out_channels,
                             x.size(-2), x.size(-1) // 2 + 1,
                             dtype=torch.cfloat, device=x.device)


        out_ft[:, :, :self.modes1, :self.modes2] = self.compl_mul2d(x_ft[:, :, :self.modes1, :self.modes2],
                             self.weights1)
        out_ft[:, :, -self.modes1:, :self.modes2] = self.compl_mul2d(x_ft[:, :, -self.modes1:, :self.modes2],
                             self.weights2)

        ## INVERSE FFT
        x = torch.fft.irfft2(out_ft, s=(x.size(-2), x.size(-1)))
        return x

In [ ]:
class iDAFNO2d(nn.Module):
    def __init__(self, modes1, modes2, width, nlayer,
                 inp_size=5, out_size=5):
        super(iDAFNO2d, self).__init__()


        self.modes1   = modes1
        self.modes2   = modes2
        self.width    = width
        self.nlayer   = nlayer
        self.inp_size = inp_size
        self.out_size = out_size


        self.fc0 = nn.Linear(self.inp_size, self.width)


        self.convlayer = nn.ModuleList([
            SpectralConv2d(self.width, self.width,
                           self.modes1, self.modes2)
            for _ in range(1)
        ])


        self.w = nn.ModuleList([
            nn.Conv2d(self.width, self.width, 1)
            for _ in range(1)
        ])

        self.fc1 = nn.Sequential(
            nn.Linear(self.width, 128),
            nn.GELU(),
            nn.Linear(128, self.out_size)
        )

    def forward(self, x, chi):

        batchsize = x.shape[0]
        size_x    = x.shape[2]
        size_y    = x.shape[3]


        chi_expand = chi.expand(batchsize, self.width, size_x, size_y) #(b,32,64,64)


        x = x.permute(0, 2, 3, 1) #(b,64,64,5)


        x = self.fc0(x) #(b,64,64,32)


        x = x.permute(0, 3, 1, 2) #(b,32,64,64)

        coef     = 1.0 / self.nlayer
        conv_chi = self.convlayer[0](chi_expand)



        for _ in range(self.nlayer - 1):


            conv_chix = self.convlayer[0](chi_expand * x)


            xconv_chi = x * conv_chi


            wx = self.w[0](x)


            x = F.gelu(chi_expand * (conv_chix - xconv_chi + wx)) * coef + x


        conv_chix = self.convlayer[0](chi_expand * x)
        xconv_chi = x * conv_chi
        wx        = self.w[0](x)


        x = chi_expand * (conv_chix - xconv_chi + wx) * coef + x


        x = x.permute(0, 2, 3, 1)


        x = self.fc1(x)


        x = x.permute(0, 3, 1, 2)


        return x



##Physics Functions

In [ ]:
def finite_difference_derivative(field, direction):

    if direction == 'x':
        dim = 2
        N   = field.shape[2]
    else:
        dim = 1
        N   = field.shape[1]

    dx = 1.0 / (N - 1)
    d  = torch.zeros_like(field)

    if direction == 'x':
        d[:, :, 1:-1] = (field[:, :, 2:] - field[:, :, :-2]) / (2 * dx)
        d[:, :, 0]    = (field[:, :, 1]  - field[:, :, 0])   / dx
        d[:, :, -1]   = (field[:, :, -1] - field[:, :, -2])  / dx
    else:
        d[:, 1:-1, :] = (field[:, 2:, :] - field[:, :-2, :]) / (2 * dx)
        d[:, 0,    :] = (field[:, 1,  :] - field[:, 0,   :]) / dx
        d[:, -1,   :] = (field[:, -1, :] - field[:, -2,  :]) / dx

    return d

In [ ]:
def compute_euler_residuals(fields_phys, clamp_val=10.0, gamma=1.4):
    gamma=1.4

    ro  = fields_phys[:, 0]
    rou = fields_phys[:, 1]
    rov = fields_phys[:, 2]
    roe = fields_phys[:, 3]


    ro_safe = ro.abs()  + 1e-8
    u       = rou / ro_safe
    v       = rov / ro_safe
    p       = (gamma - 1.0) * (roe - 0.5*(rou**2 + rov**2) / ro_safe)
    p_safe  = p.abs() + 1e-8


    R_mass = (finite_difference_derivative(rou, 'x') +
              finite_difference_derivative(rov, 'y'))


    R_momx = (finite_difference_derivative(rou**2/ro_safe + p_safe, 'x') +
              finite_difference_derivative(rou*rov/ro_safe,          'y'))


    R_momy = (finite_difference_derivative(rou*rov/ro_safe,          'x') +
              finite_difference_derivative(rov**2/ro_safe + p_safe,  'y'))


    R_ene  = (finite_difference_derivative((roe + p_safe) * u, 'x') +
              finite_difference_derivative((roe + p_safe) * v, 'y'))


    R_mass = torch.clamp(R_mass, -clamp_val, clamp_val)
    R_momx = torch.clamp(R_momx, -clamp_val, clamp_val)
    R_momy = torch.clamp(R_momy, -clamp_val, clamp_val)
    R_ene  = torch.clamp(R_ene,  -clamp_val, clamp_val)

    return R_mass, R_momx, R_momy, R_ene

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def precompute_RS_targets(Y_phys_np, batch_size=32, device=device):

    Y_tensor = torch.tensor(Y_phys_np, dtype=torch.float32).permute(0,3,1,2)
    N        = Y_tensor.shape[0]
    lists    = [[], [], [], []]

    for i in range(0, N, batch_size):
        batch = Y_tensor[i:i+batch_size].to(device)
        with torch.no_grad():
            Rs = compute_euler_residuals(batch)
        for k, R in enumerate(Rs):
            lists[k].append(R.cpu())

    return [torch.cat(l, dim=0) for l in lists]



RS_tr_mass,  RS_tr_momx,  RS_tr_momy,  RS_tr_ene  = precompute_RS_targets(Y_tr_raw)
RS_val_mass, RS_val_momx, RS_val_momy, RS_val_ene = precompute_RS_targets(Y_val_raw)



In [ ]:
SCALE_MASS = float(RS_tr_mass.std() ** 2) + 1e-8
SCALE_MOMX = float(RS_tr_momx.std() ** 2) + 1e-8
SCALE_MOMY = float(RS_tr_momy.std() ** 2) + 1e-8
SCALE_ENE  = float(RS_tr_ene.std()  ** 2) + 1e-8



def rans_physics_loss(pred_phys,
                      rs_mass_gt,
                      rs_momx_gt,
                      rs_momy_gt,
                      rs_ene_gt,
                      w_mass    = 0.0,
                      w_momx    = 0.5,
                      w_momy    = 0.5,
                      w_ene     = 2.0,
                      clamp_val = 10.0):

    R_mass_p, R_momx_p, R_momy_p, R_ene_p = compute_euler_residuals(
        pred_phys, clamp_val=clamp_val
    )


    def prep(t):
        return torch.clamp(t.to(pred_phys.device), -clamp_val, clamp_val)


    l_mass = ((R_mass_p - prep(rs_mass_gt))**2).mean() / SCALE_MASS
    l_momx = ((R_momx_p - prep(rs_momx_gt))**2).mean() / SCALE_MOMX
    l_momy = ((R_momy_p - prep(rs_momy_gt))**2).mean() / SCALE_MOMY
    l_ene  = ((R_ene_p  - prep(rs_ene_gt ))**2).mean() / SCALE_ENE

    loss_phys = (w_mass * l_mass +
                 w_momx * l_momx +
                 w_momy * l_momy +
                 w_ene  * l_ene)

    return loss_phys, l_mass, l_momx, l_momy, l_ene



In [ ]:
class CFDGridDataset(Dataset):
    def __init__(self, X, Y, Chi, rs_mass=None, rs_momx=None,
                 rs_momy=None, rs_ene=None):

        self.X   = torch.tensor(X,   dtype=torch.float32).permute(0,3,1,2)
        self.Y   = torch.tensor(Y,   dtype=torch.float32).permute(0,3,1,2)
        self.Chi = torch.tensor(Chi, dtype=torch.float32).unsqueeze(1)

        N, H, W = self.X.shape[0], self.X.shape[2], self.X.shape[3]
        zeros   = torch.zeros(N, H, W)
        self.rs_mass = rs_mass if rs_mass is not None else zeros
        self.rs_momx = rs_momx if rs_momx is not None else zeros
        self.rs_momy = rs_momy if rs_momy is not None else zeros
        self.rs_ene  = rs_ene  if rs_ene  is not None else zeros

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return (self.X[idx], self.Y[idx], self.Chi[idx], self.rs_mass[idx],
                self.rs_momx[idx], self.rs_momy[idx], self.rs_ene[idx])


train_dataset = CFDGridDataset(X_tr,  Y_tr,  Chi_tr,
                               RS_tr_mass,  RS_tr_momx,
                               RS_tr_momy,  RS_tr_ene)
val_dataset   = CFDGridDataset(X_val, Y_val, Chi_val,
                               RS_val_mass, RS_val_momx,
                               RS_val_momy, RS_val_ene)

train_loader = DataLoader(train_dataset, batch_size=4,
                          shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=4,
                          shuffle=False, num_workers=0)


## Training

In [ ]:

LAMBDA_PHYS  = 1e-3
PHYS_WARMUP  = 50
PHYS_RAMP    = 50
EPOCHS       = 500
PATIENCE     = 150
GRAD_CLIP    = 1.0
SAVE_PATH    = 'best_model_vki_rans_physics.ckpt'


Y_mean_t = torch.tensor(Y_mean, dtype=torch.float32).to(device)  # (5,)
Y_std_t  = torch.tensor(Y_std,  dtype=torch.float32).to(device)  # (5,)

def decode_prediction(pred_norm):

    return pred_norm * Y_std_t[None,:,None,None] \
                     + Y_mean_t[None,:,None,None]


torch.manual_seed(42)
model = iDAFNO2d(
    modes1=16, modes2=16, width=32, nlayer=4,
    inp_size=5, out_size=5
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(), lr=1e-3, weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=1e-5
)
criterion  = LpLoss(p=2, reduction='mean')

best_val_loss = float('inf')
early_stop    = 0
history = {'train_data':[], 'train_phys':[], 'train_total':[], 'val':[]}



In [ ]:
for ep in range(EPOCHS):


    if ep < PHYS_WARMUP:
        phys_w = 0.0
    elif ep < PHYS_WARMUP + PHYS_RAMP:
        phys_w = LAMBDA_PHYS * (ep - PHYS_WARMUP) / PHYS_RAMP
    else:
        phys_w = LAMBDA_PHYS


    model.train()
    ep_data = ep_phys = ep_total = 0.0

    for x_b, y_b, chi_b, rs_mass, rs_momx, rs_momy, rs_ene in train_loader:

        x_b   = x_b.to(device)
        y_b   = y_b.to(device)
        chi_b = chi_b.to(device)

        optimizer.zero_grad()

        pred      = model(x_b, chi_b)
        loss_data = criterion(pred, y_b)

        if phys_w > 0.0:
            pred_phys = decode_prediction(pred)
            loss_phys, lm, lx, ly, le = rans_physics_loss(
                pred_phys,
                rs_mass, rs_momx, rs_momy, rs_ene
            )
            loss = loss_data + phys_w * loss_phys
        else:
            loss_phys = torch.tensor(0.0)
            loss      = loss_data

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        ep_data  += loss_data.item()
        ep_phys  += loss_phys.item()
        ep_total += loss.item()

    n        = len(train_loader)
    ep_data /= n;  ep_phys /= n;  ep_total /= n

    history['train_data' ].append(ep_data)
    history['train_phys' ].append(ep_phys)
    history['train_total'].append(ep_total)

    # VALIDATE
    model.eval()
    ep_val = 0.0
    with torch.no_grad():
        for x_b, y_b, chi_b, *_ in val_loader:
            pred    = model(x_b.to(device), chi_b.to(device))
            ep_val += criterion(pred, y_b.to(device)).item()
    ep_val /= len(val_loader)
    history['val'].append(ep_val)

    scheduler.step()


    if ep_val < best_val_loss:
        best_val_loss = ep_val
        early_stop    = 0
        torch.save({
            'epoch'      : ep,
            'model_state': model.state_dict(),
            'optim_state': optimizer.state_dict(),
            'val_loss'   : best_val_loss,
        }, SAVE_PATH)
    else:
        early_stop += 1


    if ep % 50 == 0 or ep_val == best_val_loss:
        print(
            f"Ep {ep:4d} | "
            f"Total {ep_total:.5f} | "
            f"Data {ep_data:.5f} | "
            f"Phys {ep_phys:.5f} | "
            f"Val {ep_val:.5f} | "
            f"Best {best_val_loss:.5f} | "
            f"PhysW {phys_w:.1e} | "
            f"Pat {early_stop}/{PATIENCE}"
        )

    if early_stop >= PATIENCE:
        print(f"Early stopping at epoch {ep}")
        break

print(f"\n best val loss: {best_val_loss:.6f}")


Ep    0 | Total 0.47751 | Data 0.47751 | Phys 0.00000 | Val 0.33629 | Best 0.33629 | PhysW 0.0e+00 | Pat 0/150
Ep    1 | Total 0.30667 | Data 0.30667 | Phys 0.00000 | Val 0.28573 | Best 0.28573 | PhysW 0.0e+00 | Pat 0/150
Ep    2 | Total 0.27405 | Data 0.27405 | Phys 0.00000 | Val 0.28559 | Best 0.28559 | PhysW 0.0e+00 | Pat 0/150
Ep    3 | Total 0.25818 | Data 0.25818 | Phys 0.00000 | Val 0.26295 | Best 0.26295 | PhysW 0.0e+00 | Pat 0/150
Ep    4 | Total 0.24621 | Data 0.24621 | Phys 0.00000 | Val 0.23862 | Best 0.23862 | PhysW 0.0e+00 | Pat 0/150
Ep    6 | Total 0.23400 | Data 0.23400 | Phys 0.00000 | Val 0.22793 | Best 0.22793 | PhysW 0.0e+00 | Pat 0/150
Ep    7 | Total 0.22729 | Data 0.22729 | Phys 0.00000 | Val 0.22760 | Best 0.22760 | PhysW 0.0e+00 | Pat 0/150
Ep    8 | Total 0.22477 | Data 0.22477 | Phys 0.00000 | Val 0.22153 | Best 0.22153 | PhysW 0.0e+00 | Pat 0/150
Ep   12 | Total 0.21472 | Data 0.21472 | Phys 0.00000 | Val 0.21224 | Best 0.21224 | PhysW 0.0e+00 | Pat 0/150
E

In [ ]:
# R² Evaluation



checkpoint = torch.load(SAVE_PATH, map_location=device)
model.load_state_dict(checkpoint['model_state'])


model.eval()


all_pred = []
all_gt   = []

with torch.no_grad():
    for x_b, y_b, chi_b, *_ in val_loader:
        pred = model(x_b.to(device), chi_b.to(device))
        all_pred.append(pred.cpu())
        all_gt.append(y_b.cpu())

all_pred = torch.cat(all_pred, dim=0)
all_gt   = torch.cat(all_gt,   dim=0)


Y_mean_t = torch.tensor(Y_mean, dtype=torch.float32)
Y_std_t  = torch.tensor(Y_std,  dtype=torch.float32)

pred_phys = all_pred * Y_std_t[None,:,None,None] + Y_mean_t[None,:,None,None]
gt_phys   = all_gt   * Y_std_t[None,:,None,None] + Y_mean_t[None,:,None,None]


pred_np = pred_phys.numpy()
gt_np   = gt_phys.numpy()

channels = ['ro', 'rou', 'rov', 'roe', 'mach']


r2_scores = {}
for i, name in enumerate(channels):
    p = pred_np[:, i].flatten()
    g = gt_np[:,   i].flatten()
    r2 = r2_score(g, p)
    r2_scores[name] = r2

    print(f"{name:<8} {r2:>10.4f}  ")


ro           0.9851  
rou          0.9825  
rov          0.9922  
roe          0.9899  
mach         0.9831  
